# ImageNet-1K: DeiT-III + RRLSSO formal training

A compact Colab/Colab Enterprise entry point for the maintained two-stage experiment:

1. **192 ? 192 pre-training:** 800 epochs.
2. **224 ? 224 refinement:** 20 epochs initialized from the 192px best checkpoint.

It installs the precompiled CUDA backend, progressively caches gated ImageNet WebDataset shards, runs a real smoke test, launches a detached trainer, monitors it independently every 120 seconds, and creates resumable download archives.

Run cells from top to bottom. Never paste a token into a cell that will be committed.

## 1. Get the repository

In [ ]:
from pathlib import Path
import getpass, json, os, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/Yang916-yy/LSSO.git'
BRANCH = 'main'
COLAB_ROOT = Path('/content') if Path('/content').is_dir() else Path.home()
ROOT = COLAB_ROOT / 'LSSO'

remote = subprocess.run(
    ['git', 'ls-remote', '--exit-code', '--heads', REPO_URL, BRANCH],
    capture_output=True, text=True,
)
assert remote.returncode == 0, remote.stderr
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--prune', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
os.chdir(ROOT)
for relative in (
    'experiments/imagenet_wds_train.py',
    'examples/models/deit3_rrlsso.py',
    'tools/hf_wds_stream.py',
):
    assert (ROOT / relative).is_file(), f'missing {relative}; push the new ImageNet code first'
print('repository:', ROOT)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 2. Install the matching precompiled backend

The release wheel is tied to PyTorch 2.11 and the CUDA build. This cell installs matching PyTorch only when necessary; it does not compile MathDx.

In [ ]:
nvcc = shutil.which('nvcc')
assert nvcc, 'CUDA toolkit/nvcc is unavailable'
nvcc_version = subprocess.check_output([nvcc, '--version'], text=True)
if 'release 12.8' in nvcc_version:
    CUDA_TAG = 'cu128'
    TORCH_INDEX = 'https://download.pytorch.org/whl/cu128'
    RUNTIME_WHEEL = (
        'https://github.com/Yang916-yy/LSSO/releases/download/v0.2.0/'
        'lsso_mathdx_runtime-0.2.0%2Btorch2110cu128-py3-none-linux_x86_64.whl'
    )
elif 'release 13.' in nvcc_version:
    CUDA_TAG = 'cu130'
    TORCH_INDEX = 'https://download.pytorch.org/whl/cu130'
    RUNTIME_WHEEL = (
        'https://github.com/Yang916-yy/LSSO/releases/download/v0.2.0/'
        'lsso_mathdx_runtime-0.2.0%2Btorch2110cu130-py3-none-linux_x86_64.whl'
    )
else:
    raise RuntimeError(f'Precompiled backend supports CUDA Toolkit 12.8 or 13.x, got:\n{nvcc_version}')

probe = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
    capture_output=True, text=True,
)
expected_cuda = '12.8' if CUDA_TAG == 'cu128' else '13.0'
if probe.returncode != 0 or '2.11.0' not in probe.stdout or expected_cuda not in probe.stdout:
    assert 'torch' not in sys.modules, 'Restart the runtime, then run from cell 1'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
         'torch==2.11.0', 'torchvision==0.26.0', '--index-url', TORCH_INDEX],
        check=True,
    )
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[experiments]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', RUNTIME_WHEEL], check=True)
print(subprocess.check_output(
    [sys.executable, '-c', 'import torch; print("torch", torch.__version__, "CUDA", torch.version.cuda)'],
    text=True,
).strip())

## 3. Authenticate and configure the 192px run

Directories stay shallow: shard cache under /content/imagenet-wds, checkpoints under /content/checkpoints, and archives under /content/archives. If /content lacks space but /mnt/local-scratch is writable, only the cache moves there.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or hf_token
except Exception:
    pass
if not hf_token:
    hf_token = getpass.getpass('Hugging Face token: ')
assert hf_token.startswith('hf_'), 'Expected a Hugging Face access token'
os.environ['HF_TOKEN'] = hf_token
del hf_token

content_root = Path('/content') if Path('/content').is_dir() else Path.home()
cache_parent = content_root
if shutil.disk_usage(cache_parent).free / 2**30 < 155:
    local_scratch = Path('/mnt/local-scratch')
    if local_scratch.is_dir() and os.access(local_scratch, os.W_OK):
        cache_parent = local_scratch
CACHE_ROOT = cache_parent / 'imagenet-wds'
CHECKPOINT_ROOT = content_root / 'checkpoints'
ARCHIVE_ROOT = content_root / 'archives'
for directory in (CACHE_ROOT, CHECKPOINT_ROOT, ARCHIVE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'model': 'deit3_base_patch16_192_rrlsso',
    'stage': 'pretrain',
    'image_size': 192,
    'epochs': 800,
    'rank': 32,
    'batch_size': 768,
    'eval_batch_size': 768,
    'grad_accum': 1,
    'workers': 32,
    'eval_workers': 4,
    'seed': 0,
    'cache_dir': str(CACHE_ROOT),
    'output': str(CHECKPOINT_ROOT / 'deit3_base_rrlsso_pre192'),
    'init_checkpoint': '',
}
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))
print('cache filesystem:', CACHE_ROOT)

## 4. Check registration, GPU, disk, and native backend

In [ ]:
import gc
import torch, timm
import examples.models
from lsso.mathdx_backend import is_mathdx_available, mathdx_load_error

assert torch.cuda.is_available(), 'CUDA is unavailable'
assert timm.is_model(CONFIG['model']), f"model is not registered: {CONFIG['model']}"
assert is_mathdx_available(), f'precompiled backend failed to load: {mathdx_load_error()}'

free_gib = shutil.disk_usage(CACHE_ROOT).free / 2**30
cached_gib = sum(p.stat().st_size for p in CACHE_ROOT.glob('*.tar')) / 2**30
assert free_gib + cached_gib >= 150, (
    f'Need roughly 150 GiB for the complete cache; free={free_gib:.1f}, cached={cached_gib:.1f} GiB'
)
gpu_name = torch.cuda.get_device_name()
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
model = timm.create_model(CONFIG['model'], img_size=32, num_classes=1000, rank=CONFIG['rank'])
parameters = sum(p.numel() for p in model.parameters())
del model
gc.collect()
print(f'GPU: {gpu_name} ({gpu_gib:.1f} GiB)')
print(f'model: {CONFIG["model"]} | parameters: {parameters:,} | rank: {CONFIG["rank"]}')
print(f'disk: {free_gib:.1f} GiB free + {cached_gib:.1f} GiB cached')
print('backend: precompiled MathDx/CUDA ABI 1 loaded')
print('transport: one HTTP request + one locked cache owner per shard; no prefetch manager')

## 5. Real two-step smoke

This downloads at most one train shard and one validation shard, then checks decoding, BF16 forward/backward, native dispatch, validation, and checkpoint writing.

In [ ]:
def train_command(config, *, output=None, smoke=False, resume=True):
    command = [
        sys.executable, '-u', 'experiments/imagenet_wds_train.py',
        '--model', config['model'],
        '--stage', config['stage'],
        '--image-size', str(config['image_size']),
        '--epochs', str(config['epochs']),
        '--rank', str(config['rank']),
        '--cache-dir', config['cache_dir'],
        '--output', str(output or config['output']),
        '--batch-size', str(config['batch_size']),
        '--eval-batch-size', str(config['eval_batch_size']),
        '--grad-accum', str(config['grad_accum']),
        '--workers', str(config['workers']),
        '--eval-workers', str(config['eval_workers']),
        '--seed', str(config['seed']),
        '--require-mathdx',
        '--resume' if resume else '--no-resume',
    ]
    if config.get('init_checkpoint'):
        command += ['--init-checkpoint', config['init_checkpoint']]
    if smoke:
        command += [
            '--epochs', '1', '--steps-per-epoch', '2', '--max-val-steps', '2',
            '--batch-size', '8', '--eval-batch-size', '8',
            '--workers', '0', '--eval-workers', '0', '--shard-limit', '1',
            '--shuffle-buffer', '128',
        ]
    return command

smoke_output = Path(CONFIG['output'] + '_smoke')
started = time.time()
subprocess.run(
    train_command(CONFIG, output=smoke_output, smoke=True, resume=False),
    cwd=ROOT, env=os.environ.copy(), check=True,
)
assert (smoke_output / 'last.pt').is_file()
print(f'smoke passed in {time.time() - started:.1f}s:', smoke_output)

## 6. Launch or resume detached training

Run once. Existing last.pt restores model, optimizer, epoch, global update, LR progress, and saved RNG states. The exact WebDataset cursor is intentionally not restored.

In [ ]:
output_dir = Path(CONFIG['output'])
output_dir.mkdir(parents=True, exist_ok=True)
pid_path = output_dir / 'trainer.pid'
log_path = output_dir / 'train.log'

def trainer_is_live(pid):
    try:
        state = Path(f'/proc/{pid}/stat').read_text().split()[2]
        cmdline = Path(f'/proc/{pid}/cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
        return state != 'Z' and 'experiments/imagenet_wds_train.py' in cmdline and str(output_dir) in cmdline
    except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError, IndexError):
        return False

if pid_path.is_file():
    old_pid = int(pid_path.read_text().strip())
    if trainer_is_live(old_pid):
        raise RuntimeError(f'trainer PID {old_pid} is already running')
    pid_path.unlink(missing_ok=True)

log = log_path.open('a', buffering=1)
process = subprocess.Popen(
    train_command(CONFIG, resume=True),
    cwd=ROOT, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT,
    start_new_session=True, close_fds=True,
)
log.close()
pid_path.write_text(str(process.pid))
time.sleep(3)
if process.poll() is not None:
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-50:]))
    raise RuntimeError(f'trainer exited during startup: {process.returncode}')
print('trainer started:', process.pid)
print('output:', output_dir)
print('log:', log_path)
print('Interrupting the independent monitor below does not stop training.')

## 7. Independent 120-second monitor

In [ ]:
from IPython.display import clear_output

monitor_output = Path(CONFIG['output'])
monitor_cache = Path(CONFIG['cache_dir'])

def find_trainers():
    matches = []
    for proc_dir in Path('/proc').glob('[0-9]*'):
        try:
            cmdline = proc_dir.joinpath('cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
            stat = proc_dir.joinpath('stat').read_text().split()
            pid, parent, state = int(proc_dir.name), int(stat[3]), stat[2]
        except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError, IndexError):
            continue
        if 'experiments/imagenet_wds_train.py' in cmdline and str(monitor_output) in cmdline and state != 'Z':
            matches.append((pid, parent, state))
    pids = {pid for pid, _, _ in matches}
    roots = [(pid, state) for pid, parent, state in matches if parent not in pids]
    return roots, len(matches) - len(roots)

def tail(path, lines):
    path = Path(path)
    if not path.is_file():
        return '(not created yet)'
    content = path.read_text(errors='replace').splitlines()
    return '\n'.join(content[-lines:]) if content else '(empty)'

try:
    while True:
        clear_output(wait=True)
        print(time.strftime('%Y-%m-%d %H:%M:%S'))
        trainers, loader_workers = find_trainers()
        print('trainer:', trainers if trainers else 'not found')
        print('DataLoader workers:', loader_workers)
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw',
            '--format=csv,noheader',
        ], check=False)
        shards = list(monitor_cache.glob('*.tar'))
        partials = list(monitor_cache.glob('*.partial'))
        shard_gib = sum(p.stat().st_size for p in shards) / 2**30
        partial_gib = sum(p.stat().st_size for p in partials) / 2**30
        print(f'HF cache: {len(shards)} complete ({shard_gib:.2f} GiB), '
              f'{len(partials)} active ({partial_gib:.2f} GiB)')
        print('--- metrics ---')
        print(tail(monitor_output / 'metrics.csv', 6))
        print('--- log tail ---')
        print(tail(monitor_output / 'train.log', 20))
        last = monitor_output / 'last.pt'
        if last.is_file():
            updated = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(last.stat().st_mtime))
            print(f'last.pt: {last.stat().st_size / 2**30:.2f} GiB, updated {updated}')
        else:
            print('last.pt: not created yet (saved after the first complete epoch)')
        print('next refresh in 120 seconds')
        for _ in range(24):
            time.sleep(5)
except KeyboardInterrupt:
    print('Monitor stopped; trainer was not signaled.')

## 8. Create and download a resume archive

Stop only the monitor cell first. Training may continue while this takes a stable copy of last.pt. best.pt is excluded by default to keep the archive small.

In [ ]:
from datetime import datetime
import gc, zipfile
from google.colab import files

run_dir = Path(CONFIG['output'])
last_source = run_dir / 'last.pt'
assert last_source.is_file(), 'finish at least one epoch first'
INCLUDE_BEST = False
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
stage_dir = ARCHIVE_ROOT / f'{run_dir.name}-{stamp}'
archive_path = ARCHIVE_ROOT / f'{run_dir.name}-{stamp}.zip'
stage_dir.mkdir(parents=True, exist_ok=False)

def stable_copy(source, destination, attempts=10):
    for _ in range(attempts):
        before = source.stat()
        signature = before.st_size, before.st_mtime_ns
        shutil.copy2(source, destination)
        after = source.stat()
        if signature == (after.st_size, after.st_mtime_ns) and destination.stat().st_size == after.st_size:
            return
        destination.unlink(missing_ok=True)
        time.sleep(3)
    raise RuntimeError(f'{source.name} kept changing; retry between checkpoint writes')

try:
    stable_copy(last_source, stage_dir / 'last.pt')
    if INCLUDE_BEST and (run_dir / 'best.pt').is_file():
        stable_copy(run_dir / 'best.pt', stage_dir / 'best.pt')
    for name in ('config.json', 'metrics.csv', 'train.log'):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, stage_dir / name)
    checkpoint = torch.load(stage_dir / 'last.pt', map_location='cpu', weights_only=False, mmap=True)
    assert {'model', 'optimizer', 'epoch', 'global_update'} <= checkpoint.keys()
    print('validated:', {key: checkpoint.get(key) for key in ('epoch', 'global_update', 'best_acc')})
    del checkpoint
    gc.collect()
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
        for source in sorted(stage_dir.iterdir()):
            bundle.write(source, arcname=f'{run_dir.name}/{source.name}')
    with zipfile.ZipFile(archive_path) as bundle:
        assert bundle.testzip() is None
finally:
    shutil.rmtree(stage_dir, ignore_errors=True)
print(f'archive: {archive_path} ({archive_path.stat().st_size / 2**30:.2f} GiB)')
files.download(str(archive_path))

## 9. Switch to 224px refinement after pre-training

Run only after the 192px stage finishes. It changes CONFIG to a separate shallow directory. Then rerun cells 5?8.

In [ ]:
pretrain_dir = CHECKPOINT_ROOT / 'deit3_base_rrlsso_pre192'
pretrain_best = pretrain_dir / 'best.pt'
assert pretrain_best.is_file(), f'missing {pretrain_best}'
CONFIG.update({
    'stage': 'finetune',
    'image_size': 224,
    'epochs': 20,
    'batch_size': 512,
    'eval_batch_size': 512,
    'output': str(CHECKPOINT_ROOT / 'deit3_base_rrlsso_ft224'),
    'init_checkpoint': str(pretrain_best),
})
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))
print('Rerun cells 5?8. The trainer interpolates learned PE from 12x12 to 14x14.')

## Notes

- For Small or Large, change the registered model name and output directory before smoke.
- Never launch two sizes simultaneously on one GPU.
- Stopping the monitor does not signal the detached trainer.
- The complete cache has 1,024 train shards and 64 validation shards and is shared by both stages.
- Resume restores optimization state, but the randomized streaming sample order after restart is newly stochastic.